# 02 Results (`:CALCULATED`)
*Run 2026-11-17 with `uncertainty_engine.py`. Part A: the Ask-Holcus predictions HP1–HP4 (`01_holcus_asks.ipynb`, registered before this ran). Part B: my own engine self-checks P1–P7 (`01b_engine_selfchecks.ipynb`). Nothing is tuned.*

# Part A: Ask Holcus (HP1–HP4)
## HP1: the same circle under a different f

In [1]:
import uncertainty_engine as U, numpy as np
for N in (600, 1200, 2400):
    r = U.hp1_other_hamiltonian(N=N)
    print('N=%4d  min overlap %.10f   sigma_x sigma_p error: n=1 %.2e, n=5 %.2e' % (N, min(x['overlap'] for x in r), abs(r[1]['sigma_x_sigma_p']-1.5), abs(r[5]['sigma_x_sigma_p']-5.5)))
for x in U.hp1_other_hamiltonian(N=2400): print('  n=%d  E=%.4f  (n+1/2)^2=%.4f  overlap %.10f' % (x['n'], x['E'], x['E_expected_(n+1/2)^2'], x['overlap']))

N= 600  min overlap 0.9999991184   sigma_x sigma_p error: n=1 8.43e-04, n=5 1.03e-02


N=1200  min overlap 0.9999999452   sigma_x sigma_p error: n=1 2.10e-04, n=5 2.57e-03


N=2400  min overlap 0.9999999966   sigma_x sigma_p error: n=1 5.26e-05, n=5 6.41e-04


  n=0  E=0.2500  (n+1/2)^2=0.2500  overlap 1.0000000000
  n=1  E=2.2500  (n+1/2)^2=2.2500  overlap 0.9999999999
  n=2  E=6.2498  (n+1/2)^2=6.2500  overlap 0.9999999998
  n=3  E=12.2495  (n+1/2)^2=12.2500  overlap 0.9999999993
  n=4  E=20.2490  (n+1/2)^2=20.2500  overlap 0.9999999983
  n=5  E=30.2482  (n+1/2)^2=30.2500  overlap 0.9999999966


**HP1, overlap clause holds** (min overlap 1 − 3e-9 at N = 2400; threshold 1 − 1e-6): the eigenstates of H = (p² + x²)²/4 are the oscillator's own functions, and the energies are (n + ½)², as they must be since this H is the square of the oscillator Hamiltonian. **HP1, σₓσₚ clause misses its threshold at the grids tried** (error 6.4e-4 for n = 5 at N = 2400, against the 1e-4 set): the error falls by 4 for every doubling of N (1.0e-2, 2.6e-3, 6.4e-4), which is the second-order convergence of the finite-difference operator, not a property of the states; the threshold was set without counting what a second-order grid can reach. Recorded as a miss, with the cause. What the viewport said is borne out: a different f changes the speed (the energies), not the circle (the functions).

## HP2: the minimum cell from the zeros of ζ

In [2]:
g = np.loadtxt('../../RiemannHypothesisProof/data/zeros_1_to_3600.csv', comments='#')
r = U.hp2_cell_fit(g)
print('%d zeros, E from 100 to 3500: fitted c = %.5f   slope %.2e   intercept %.3f (Berry-Keating: 7/8 = 0.875)   residual rms %.3f' % (r['n_zeros'], r['c'], r['slope'], r['intercept'], r['residual_rms']))
for lo, hi in ((100,1000),(1000,2000),(2000,3500)): print('  window %4d-%4d  c = %.4f' % (lo, hi, U.hp2_cell_fit(g, lo, hi, 200)['c']))
print('c = 2 would drift at %.3f E (about %.0f zeros at E = 3500); c = 1/2 at %.3f E' % (r['drift_excluded_c2'], abs(r['drift_excluded_c2'])*3500, r['drift_excluded_chalf']))

3066 zeros, E from 100 to 3500: fitted c = 1.00008   slope -1.27e-05   intercept 0.894 (Berry-Keating: 7/8 = 0.875)   residual rms 0.396
  window  100-1000  c = 0.9990
  window 1000-2000  c = 1.0005
  window 2000-3500  c = 1.0002
c = 2 would drift at -0.110 E (about 386 zeros at E = 3500); c = 1/2 at 0.110 E


**HP2 holds.** The count of the zeros of ζ below E equals the phase-space area of {xp ≤ E, x > lₓ, p > lₚ} in units of h exactly when the minimum cell lₓlₚ is **one** quantum h: the fit gives c = 1.00008 (threshold 0.01), stable across three windows (0.9990, 1.0005, 1.0002), with an intercept of 0.894 against the Berry–Keating 7/8 = 0.875 (residual rms 0.40, so within noise). c = 2 or ½ would drift by 0.11·E, about 390 zeros at E = 3500; the residual is 0.40. So the quantum of phase-space area that the uncertainty relation names is the cell that makes xp count the primes' zeros: not negotiable at c = 1 to 1e-3. This is the Berry–Keating count (`ESTABLISHED`) read the other way: the cell is an input and the data fix it. What it means for deriving the cell is `OPEN`.

## HP3 and HP4

In [3]:
x, dx = U.grid()
print('HP3  max |FT(psi_n) - (-i)^n psi_n|:', ['%.1e' % abs(U.fourier_pair(U.norm(U.hermite_state(n, x), dx), x, dx) - (-1j)**n*U.norm(U.hermite_state(n, x), dx)).max() for n in range(6)])
print('HP4  sampled-orbit area / (2 pi E):', ['%.9f' % (U.hp4_orbit_area(E)/(2*np.pi*E)) for E in (0.5, 1.5, 4.5)])

HP3  max |FT(psi_n) - (-i)^n psi_n|: ['6.7e-16', '6.8e-16', '5.6e-16', '6.8e-16', '5.6e-16', '7.8e-16']
HP4  sampled-orbit area / (2 pi E): ['0.999999999', '0.999999999', '0.999999999']


**HP3 holds** (7e-16 against 1e-8). **HP4 holds** (1 − 1e-9): the circle's area is 2πE; it is exact geometry. The Lagrangian viewport says scale and total derivative are free; the area of the circle at a given energy is not. Where the *quantum* of that area comes from is the open question.

In [4]:
import uncertainty_engine as U, numpy as np
x, dx = U.grid()
print('grid', len(x), 'points, dx = %.4f' % dx)

grid 2048 points, dx = 0.0137


# Part B: engine self-checks P1–P7 (mine, not Holcus predictions)

## P1, P2: eigenstates and the circle of the same energy

In [5]:
for n in range(9):
    psi = U.norm(U.hermite_state(n, x), dx); sx, sp = U.spreads(psi, x, dx); c = U.circle_of_energy(n + 0.5)
    print('n=%d  sigma_x*sigma_p = %.9f  (n+1/2) = %.1f  circle: rms_x*rms_p = %.9f, area/h = %.9f' % (n, sx*sp, n+.5, c['rms_x']*c['rms_p'], c['area']/(2*np.pi)))

n=0  sigma_x*sigma_p = 0.500000000  (n+1/2) = 0.5  circle: rms_x*rms_p = 0.500000000, area/h = 0.500000000
n=1  sigma_x*sigma_p = 1.500000000  (n+1/2) = 1.5  circle: rms_x*rms_p = 1.500000000, area/h = 1.500000000
n=2  sigma_x*sigma_p = 2.500000000  (n+1/2) = 2.5  circle: rms_x*rms_p = 2.500000000, area/h = 2.500000000
n=3  sigma_x*sigma_p = 3.500000000  (n+1/2) = 3.5  circle: rms_x*rms_p = 3.500000000, area/h = 3.500000000
n=4  sigma_x*sigma_p = 4.500000000  (n+1/2) = 4.5  circle: rms_x*rms_p = 4.500000000, area/h = 4.500000000
n=5  sigma_x*sigma_p = 5.500000000  (n+1/2) = 5.5  circle: rms_x*rms_p = 5.500000000, area/h = 5.500000000
n=6  sigma_x*sigma_p = 6.500000000  (n+1/2) = 6.5  circle: rms_x*rms_p = 6.500000000, area/h = 6.500000000
n=7  sigma_x*sigma_p = 7.500000000  (n+1/2) = 7.5  circle: rms_x*rms_p = 7.500000000, area/h = 7.500000000
n=8  sigma_x*sigma_p = 8.500000000  (n+1/2) = 8.5  circle: rms_x*rms_p = 8.500000000, area/h = 8.500000000


**P1 and P2 hold** to 9 digits for n = 0…8 (threshold 1e-6): the quantum spreads of an eigenstate equal the projections of the classical circle of the same energy, and the circle's area is (n + ½)h. This is the Bohr–Sommerfeld statement with the half-quantum in it; it is `ESTABLISHED` that the numbers agree, and it is the content of the claim that one circle sits behind both readings.

## P3: no preferred angle

In [6]:
for n in (0, 1, 4):
    psi = U.norm(U.hermite_state(n, x), dx)
    s = np.array([U.quadrature_spread(psi, x, dx, th) for th in np.linspace(0, np.pi, 37)])
    print('n=%d  spread at 37 angles: min %.9f max %.9f  relative spread %.1e' % (n, s.min(), s.max(), (s.max()-s.min())/s.mean()))

n=0  spread at 37 angles: min 0.707106781 max 0.707106781  relative spread 1.6e-16
n=1  spread at 37 angles: min 1.224744871 max 1.224744871  relative spread 1.8e-16
n=4  spread at 37 angles: min 2.121320344 max 2.121320344  relative spread 0.0e+00


**P3 holds** (1e-16). Position and momentum are two angles of one thing: an eigenstate gives the same spread at every angle, because its orbit is a circle.

## P4: a state that does prefer an angle (squeezed Gaussian)

In [7]:
psi = U.norm(U.gaussian_state(0.5, x), dx); sx, sp = U.spreads(psi, x, dx)
print('sigma_x %.4f  sigma_p %.4f  product %.4f' % (sx, sp, sx*sp))
for th in (0, np.pi/8, np.pi/4, 3*np.pi/8, np.pi/2):
    a = U.quadrature_spread(psi, x, dx, th); b = U.quadrature_spread(psi, x, dx, th + np.pi/2)
    f = np.sqrt(sx**2*np.cos(th)**2 + sp**2*np.sin(th)**2)
    print('theta=%.3f  spread %.6f  ellipse formula %.6f  conjugate product %.6f' % (th, a, f, a*b))

sigma_x 0.5000  sigma_p 1.0000  product 0.5000
theta=0.000  spread 0.500000  ellipse formula 0.500000  conjugate product 0.500000
theta=0.393  spread 0.599862  ellipse formula 0.599862  conjugate product 0.565962
theta=0.785  spread 0.790569  ellipse formula 0.790569  conjugate product 0.625000
theta=1.178  spread 0.943486  ellipse formula 0.943486  conjugate product 0.565962
theta=1.571  spread 1.000000  ellipse formula 1.000000  conjugate product 0.500000


**P4, first clause holds:** the spread follows the ellipse σ_θ² = σₓ²cos²θ + σₚ²sin²θ to 6 digits (threshold 1e-6), and the conjugate-angle product is never below ħ/2.

**P4, second clause fails, and the prediction was wrong, not the engine.** As committed, P4 said the conjugate product equals ħ/2 "at every θ for a pure Gaussian". It does not: it is 0.5000 at θ = 0 and π/2, and 0.5660 and 0.6250 at θ = π/8 and π/4. The product is (σₓ²cos²θ + σₚ²sin²θ)(σₓ²sin²θ + σₚ²cos²θ), which exceeds σₓ²σₚ² unless σₓ = σₚ or θ is a multiple of π/2. Only a circular (σₓ = σₚ) state gives ħ/2 at every angle. The committed text is kept; this is the correction. The corrected, and sharper, statement is the content of the claim: the *minimum over angles of the conjugate product* is ħ/2 and it sits on the axes the state is squeezed along; the flattening onto the axes is where the relation looks tight.

## P5: motion = 0 costs unbounded motion

In [8]:
for s in (1.0, 0.5, 0.25, 0.1):
    p = U.norm(U.gaussian_state(s, x), dx); E = 0.5*np.sum(abs(U.apply_p(p, dx))**2)*dx
    print('sigma=%.2f  KE %.6f  hbar^2/(8 m sigma^2) %.6f' % (s, E, 1/(8*s**2)))

sigma=1.00  KE 0.125000  hbar^2/(8 m sigma^2) 0.125000
sigma=0.50  KE 0.500000  hbar^2/(8 m sigma^2) 0.500000
sigma=0.25  KE 2.000000  hbar^2/(8 m sigma^2) 2.000000
sigma=0.10  KE 12.500000  hbar^2/(8 m sigma^2) 12.500000


**P5 holds** (exact). Squeezing the position spread toward a point (motion = 0 in the position reading) raises the kinetic energy as 1/σ²: the point is paid for with unbounded motion in the other reading.

## P7: the Fourier pair as a quarter turn

In [9]:
for n in range(6):
    psi = U.norm(U.hermite_state(n, x), dx); phi = U.fourier_pair(psi, x, dx)
    print('n=%d  max |FT(psi_n) - (-i)^n psi_n| = %.2e' % (n, abs(phi - (-1j)**n*psi).max()))

n=0  max |FT(psi_n) - (-i)^n psi_n| = 6.66e-16


n=1  max |FT(psi_n) - (-i)^n psi_n| = 6.78e-16


n=2  max |FT(psi_n) - (-i)^n psi_n| = 5.55e-16


n=3  max |FT(psi_n) - (-i)^n psi_n| = 6.75e-16


n=4  max |FT(psi_n) - (-i)^n psi_n| = 5.59e-16


n=5  max |FT(psi_n) - (-i)^n psi_n| = 7.78e-16


**P7 holds** (7e-16). The transform is a quarter turn of the (x, p) plane: eigenstate n picks up the phase (−i)ⁿ, a rotation by nπ/2 of the phase of its own circle.

## P6: the video's helium estimate

In [10]:
h = U.helium_estimates()
for k, v in h.items(): print('%-26s %.4f' % (k, v))
measured = 14.0   # K per atom, recalled from neutron-scattering and Monte Carlo work: to be checked against a citation
print('measured kinetic energy per atom (recalled) ~ %.1f K' % measured)
print('video estimate is a factor %.1f below; 3-D Gaussian with sigma = r/2 is a factor %.1f below' % (measured/h['video_E_K'], measured/h['gauss3d_sigma_r_over_2_K']))

hbar2_over_2m_K_A2         6.0596
r_A                        2.2197
video_E_K                  1.2299
gauss3d_sigma_r_over_2_K   3.6898
measured kinetic energy per atom (recalled) ~ 14.0 K
video estimate is a factor 11.4 below; 3-D Gaussian with sigma = r/2 is a factor 3.8 below


**P6 fails** (threshold: within a factor of 3). The video's estimate, with r the radius of the volume per atom (2.22 Å), gives 1.23 K per atom, about 11 times below the measured kinetic energy per atom; a three-dimensional Gaussian with σ = r/2 gives 3.7 K, about 4 times below. The order of magnitude and the reason (confinement forces motion, so helium cannot be locked still) are right; the number is not, because r is the volume per atom and not the cage the atom actually moves in. The measured value used here is recalled (about 14 K) and is **to be checked** against a citation before any use. Failure kept.

**Summary.** P1, P2, P3, P5, P7 hold. P4 holds in its first clause; its second clause was a wrong prediction, corrected above. P6 fails. Nothing here derives the minimum circle without [x, p] = iħ: the paper's claim that the uncertainty relation need not be a separate postulate is **`OPEN`**.